# 04 · Explainability and ablations

Each section starts by restoring the run it builds on, so the sections are independent and can run in any order (or in separate sessions).

**Kaggle settings:** Accelerator *GPU T4 x2* (the code uses one GPU), Internet *on*,
secret `HF_TOKEN` (a Hugging Face token with write access) switched on for this notebook.
**Inputs:** APTOS 2019 Blindness Detection (competition); IDRiD (`aaryapatel98/indian-diabetic-retinopathy-image-dataset`).

`%%bash` must be the first line of its cell; put notes in markdown cells, not above it.

In [ ]:
import os, json
os.chdir('/kaggle/working')
if not os.path.exists('retina-screen'):
    !git clone -q https://github.com/ashwinruke/retina-screen.git
%cd /kaggle/working/retina-screen
!git pull -q
!pip install -q timm albumentations onnx onnxscript onnxruntime
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
!nvidia-smi --query-gpu=name --format=csv,noheader

In [ ]:
def show(*names):
    for name in names:
        r = json.load(open(f'outputs/eval/{name}.json'))['tuned_thresholds']
        rec = r['per_grade_recall']
        print(f"{name:20s} n={r['n']:4d} QWK={r['qwk']:.3f} AUC={r['referable']['auc']:.3f} "
              f"sens={r['referable']['sensitivity']:.3f} spec={r['referable']['specificity']:.3f} "
              f"severe={rec['Severe NPDR']:.2f} prolif={rec['Proliferative DR']:.2f} "
              f"cuts={[round(t, 2) for t in r['thresholds']]}")

## Grad-CAM vs clinicians' lesion masks (A1, ~5 min)
Pointing game and heat share on IDRiD's 81 segmentation photos, plus the gallery figure.

In [ ]:
!python scripts/artifacts.py pull --tag a1
!python scripts/explain.py --model outputs/model.pt \
    --grading-manifest data/manifests/idrid.csv --seg-root auto \
    --out-dir docs/figures --prefix a1 --report outputs/eval/explain_a1.json
!python scripts/artifacts.py push --tag a1 --run pretrain_eyepacs finetune_aptos

## A4 · Adapting to IDRiD with local data (~10 min)
IDRiD's training images split 330 / 83. A4a refits only the cut points on the 83; A4 also fine-tunes on the 330. Both are scored on the 103 official test images.

In [ ]:
!python scripts/artifacts.py pull --tag a1

In [ ]:
%%bash
set -e
cd /kaggle/working/retina-screen
python scripts/make_local_split.py --manifest data/manifests/idrid.csv --out data/manifests/idrid_local.csv
python scripts/cache_images.py --manifest data/manifests/idrid_local.csv --out-dir cache --config configs/finetune_idrid.yaml
python scripts/evaluate.py --checkpoint outputs/model.pt --manifest cache/idrid_local.csv --splits test --name A1_idrid_test_check
python scripts/evaluate.py --checkpoint outputs/model.pt --manifest cache/idrid_local.csv --splits test \
    --fit-manifest cache/idrid_local.csv --fit-splits val --name A4a_idrid_test
python scripts/train.py --config configs/finetune_idrid.yaml
python scripts/evaluate.py --checkpoint outputs/finetune_idrid/best.pt --manifest cache/idrid_local.csv --splits test \
    --fit-manifest cache/idrid_local.csv --fit-splits val --name A4_idrid_test
python scripts/artifacts.py push --tag a4 --run finetune_idrid --skip-model

In [ ]:
show('A1_idrid_test_check', 'A4a_idrid_test', 'A4_idrid_test')

## A3 · 5-class head instead of one severity score (~35 min)
Same as A1's fine-tuning from the EyePACS-pretrained backbone; only the head changes.

In [ ]:
!python scripts/artifacts.py pull --tag a1

In [ ]:
%%bash
set -e
cd /kaggle/working/retina-screen
python scripts/cache_images.py --manifest data/manifests/aptos.csv --out-dir cache --config configs/finetune_aptos.yaml
python scripts/cache_images.py --manifest data/manifests/idrid.csv --out-dir cache --config configs/finetune_aptos.yaml
python scripts/train.py --config configs/finetune_aptos.yaml experiment.name=A3_cls model.head=classification
python scripts/evaluate.py --checkpoint outputs/A3_cls/best.pt \
    --manifest cache/aptos.csv --splits test \
    --fit-manifest cache/aptos.csv --fit-splits val \
    --name A3_aptos_test --export outputs/model.pt
python scripts/evaluate.py --checkpoint outputs/model.pt --manifest cache/idrid.csv --splits all  --name A3_idrid
python scripts/evaluate.py --checkpoint outputs/model.pt --manifest cache/idrid.csv --splits test --name A3_idrid_test
python scripts/artifacts.py push --tag a3 --run A3_cls

In [ ]:
show('A1_aptos_test', 'A3_aptos_test', 'A1_idrid', 'A3_idrid', 'A1_idrid_test', 'A3_idrid_test')

## A2 · A0 without Ben Graham normalisation (~35 min)
Repeats A0's recipe exactly on plain cropped photos (separate cache).

In [ ]:
!python scripts/artifacts.py pull --tag mvp

In [ ]:
%%bash
set -e
cd /kaggle/working/retina-screen
python scripts/cache_images.py --manifest data/manifests/aptos.csv --out-dir cache_nobg --config configs/baseline_aptos.yaml --no-ben-graham
python scripts/cache_images.py --manifest data/manifests/idrid.csv --out-dir cache_nobg --config configs/baseline_aptos.yaml --no-ben-graham
python scripts/train.py --config configs/baseline_aptos.yaml experiment.name=A2_nobg data.manifest=cache_nobg/aptos.csv
python scripts/evaluate.py --checkpoint outputs/A2_nobg/best.pt \
    --manifest cache_nobg/aptos.csv --splits test \
    --fit-manifest cache_nobg/aptos.csv --fit-splits val \
    --name A2_aptos_test --export outputs/model.pt
python scripts/evaluate.py --checkpoint outputs/model.pt --manifest cache_nobg/idrid.csv --splits all  --name A2_idrid
python scripts/evaluate.py --checkpoint outputs/model.pt --manifest cache_nobg/idrid.csv --splits test --name A2_idrid_test
python scripts/artifacts.py push --tag a2 --run A2_nobg

In [ ]:
show('mvp_aptos_test', 'A2_aptos_test', 'A0_idrid', 'A2_idrid', 'A0_idrid_test', 'A2_idrid_test')